# 🧪 Phase 2: Production-Ready Preprocessing Pipeline
**Project:** Machine Fault Recognition System  
**Compliance:** Fully aligns with `pipeline.pdf` (Phases 1-10) AND `eda_insights.md` (EDA overrides spec where noted).

### ✅ Key Decisions (EDA-Justified Overrides)
| Decision | pipeline.pdf says | EDA says | We use |
|---|---|---|---|
| Duration | 10s | 11s — M1/M2 anomalies at end of 11s | **11.0s** |
| Silence trim | Apply | PROHIBITED — destroys M2 temporal structure | **Removed** |
| DC removal | Apply | Skip — DC offset is 0.000 across all files | **Removed** |
| Peak norm | Apply | PROHIBITED — erases RMS fault signal | **Removed** |
| Instance norm | Apply per-file | PROHIBITED — use global normalization | **Global stats** |
| Noise augment | Gaussian only | Gaussian + Pink noise | **Both** |

In [ ]:
!pip install -q torchaudio librosa tqdm

## 📦 Imports & Constants

In [ ]:
import os
import time
import torch
import random
import torchaudio
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ── Audio constants ───────────────────────────────────────────────────────────
TARGET_SR      = 16_000
TARGET_SECONDS = 11.0          # EDA override: preserves M1/M2 end-of-cycle failures
TARGET_SAMPLES = int(TARGET_SR * TARGET_SECONDS)   # 176,000

# ── Spectrogram constants ─────────────────────────────────────────────────────
N_MELS     = 128
N_FFT      = 1024
HOP_LENGTH = 512
F_MIN      = 50
F_MAX      = 8000
TOP_DB     = 80

# Output shape: [6, 128, 344]
# 344 = floor((176000 - 1024) / 512) + 1
T_FRAMES = (TARGET_SAMPLES - N_FFT) // HOP_LENGTH + 1   # 344
N_CHANNELS = 3   # mono × (mel + Δ + ΔΔ) — dataset is 100% mono (EDA §2)

print(f"Target tensor shape: [{N_CHANNELS}, {N_MELS}, {T_FRAMES}]")  # [3, 128, 344]
print(f"  -> {N_CHANNELS * N_MELS * T_FRAMES:,} elements per sample")

## 🛠️ Phase 1 — Load & Ingest

In [ ]:
def get_sorted_files(data_dir: str) -> list:
    """
    Phase 1 / pipeline.pdf §2.1 — CRITICAL: numerical sort, not alphabetical.
    Alphabetical sort would place 10.wav before 2.wav, corrupting results.txt order.
    """
    files = list(Path(data_dir).rglob("*.wav"))
    return sorted(files, key=lambda p: int(p.stem))


def load_audio(path: str):
    """
    Phase 1 / pipeline.pdf §2.2 — Load with channel validation.
    Guarantees exactly 2 channels out regardless of input.

    NOTE: EDA §2 confirms 100% of dataset is mono (1 channel).
    We still duplicate to [2, T] for API compatibility with the rest of the
    pipeline, but channels 0 and 1 will be identical. The delta features across
    both will also be identical — this means our effective tensor is [3, 128, 344]
    of unique information embedded in a [6, 128, 344] tensor. The model still
    trains correctly; just be aware when setting ConvNeXt in_channels.
    """
    waveform, sr = torchaudio.load(path)   # [C, T]
    if waveform.shape[0] == 1:
        waveform = waveform.repeat(2, 1)   # mono → stereo duplicate
    elif waveform.shape[0] > 2:
        waveform = waveform[:2, :]         # keep first two channels only
    return waveform, sr

## 🔇 Phase 2 — Resampling
> **EDA §8 override:** DC offset removal and peak normalisation have been removed.
> - DC offset is confirmed 0.0000 across all files — subtraction is a no-op that wastes time.
> - Peak normalisation would squash M3's transient "crack" spikes (amplitude > 0.04),
>   collapsing the entire baseline to near-zero. EDA §4 confirms RMS energy is a
>   statistically significant fault indicator (p=0.0000) — do not erase it.

In [ ]:
def resample_waveform(waveform: torch.Tensor, orig_sr: int) -> torch.Tensor:
    """
    Phase 4 / pipeline.pdf §5 — Resample to 16 kHz.
    Uses sinc_interp_hann for best anti-aliasing quality.
    Preserves 0-8 kHz band sufficient for mechanical fault detection.
    """
    if orig_sr == TARGET_SR:
        return waveform
    resampler = torchaudio.transforms.Resample(
        orig_freq=orig_sr,
        new_freq=TARGET_SR,
        resampling_method="sinc_interp_hann",
    )
    return resampler(waveform)

## ✂️ Phase 3 — Length Standardisation
> **EDA §2 & §6 overrides:**
> - Silence trimming is **PROHIBITED**. M2 is 86% silence and its spin-up/spin-down
>   timing IS the anomaly. Trimming would compress the audio, misalign critical
>   boundary events, and destroy temporal structure.
> - Pad M3 (10s) → 11s with trailing zeros; never truncate M1/M2 (11s).

In [ ]:
def fix_length(waveform: torch.Tensor) -> torch.Tensor:
    """
    Phase 5 (EDA override: 11s not 10s) — Pad / truncate to exactly 176,000 samples.
    - M3 (10s → 10.0s after resample) is zero-padded at the END.
    - M1/M2 (11s) pass through unchanged.
    - Padding at the END avoids shifting the critical onset events of M1/M2.
    """
    T = waveform.shape[-1]
    if T < TARGET_SAMPLES:
        waveform = torch.nn.functional.pad(waveform, (0, TARGET_SAMPLES - T))
    elif T > TARGET_SAMPLES:
        waveform = waveform[:, :TARGET_SAMPLES]
    return waveform

## 🌈 Phase 4-6 — Spectral Features (Mel + Deltas)

In [ ]:
# Build transforms once (reused across all files for speed)
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F_MIN,
    f_max=F_MAX,
    power=2.0,         # energy spectrogram
)
db_transform = torchaudio.transforms.AmplitudeToDB(
    stype="power",
    top_db=TOP_DB,     # clamp dynamic range to 80 dB
)


def compute_features(waveform: torch.Tensor) -> torch.Tensor:
    """
    Phases 7-8 / pipeline.pdf §8-9.
    Input:  [2, 176000] (or [1, 176000])
    Output: [3, 128, 344]  (mel + Δ + ΔΔ for mono channel)

    EDA §7 requirement: no smoothing/blurring of spectrograms.
    EDA §5 requirement: full 128 Mel bands (captures M1 high-freq anomalies).

    NOTE: Instance normalisation is intentionally NOT applied here.
    Global normalisation (applied after loading saved .npy files) preserves
    the absolute energy differences that are the primary fault signal (EDA §4).
    """
    # Phase 7: Mel-spectrogram + dB conversion
    # Dataset is 100% mono (EDA §2) — use channel 0 only, drop duplicate channel 1
    mel    = mel_transform(waveform[[0]])   # [1, 128, 344] — channel 0 only
    mel_db = db_transform(mel)              # log-scale

    # Phase 8: Delta features -> [3, 128, 344]
    compute_delta = torchaudio.functional.compute_deltas
    m  = mel_db[0]                          # [128, 344] mel
    d1 = compute_delta(m)                   # 1st order: rate of change
    d2 = compute_delta(d1)                  # 2nd order: acceleration

    return torch.stack([m, d1, d2])         # [3, 128, 344]

## 📊 Global Normalisation Statistics
> **EDA §4 — Critical requirement.** Per-file instance normalisation is prohibited
> because absolute RMS energy is a statistically proven fault indicator (p=0.0000).
> 
> We compute mean/std **once** from the Normal training split, save to disk,
> and apply the same statistics at both training and inference time.
> This is the ONLY normalisation used.

In [ ]:
def compute_global_stats(df_train_normal: pd.DataFrame,
                         save_path: str = "norm_stats.npz",
                         max_files: int = None) -> dict:
    """
    One-time computation of global mean and std over the Normal training set.
    Computed per-channel across the (128, 344) spatial dimensions.

    Args:
        df_train_normal: DataFrame filtered to state == 'normal', train split only.
        save_path: Where to save the stats .npz file.
        max_files: Cap for memory-limited environments (use None for full dataset).

    Returns dict with 'mean' and 'std' tensors of shape [6, 1, 1].
    """
    paths = df_train_normal['path'].tolist()
    if max_files:
        paths = paths[:max_files]

    # Welford online algorithm to avoid storing all tensors in RAM
    count = 0
    mean  = torch.zeros(N_CHANNELS, 1, 1)
    M2    = torch.zeros(N_CHANNELS, 1, 1)

    for path in tqdm(paths, desc="Computing global stats"):
        try:
            waveform, sr = load_audio(path)
            waveform     = resample_waveform(waveform, sr)
            waveform     = fix_length(waveform)
            feat         = compute_features(waveform)          # [6, 128, 344]
            # Per-channel mean over spatial dims
            x = feat.mean(dim=(-2, -1), keepdim=True)          # [6, 1, 1]
            count += 1
            delta  = x - mean
            mean  += delta / count
            M2    += delta * (x - mean)
        except Exception as e:
            print(f"  ⚠️ Skipped {path}: {e}")

    std = (M2 / (count - 1)).sqrt()
    np.savez(save_path, mean=mean.numpy(), std=std.numpy())
    print(f"✅ Global stats computed from {count} normal files → saved to {save_path}")
    return {"mean": mean, "std": std}


def load_global_stats(path: str = "norm_stats.npz") -> dict:
    """
    Load pre-computed global normalisation stats from disk.
    Call this at the start of both training AND infer.py.
    """
    data = np.load(path)
    return {
        "mean": torch.from_numpy(data["mean"]),   # [6, 1, 1]
        "std":  torch.from_numpy(data["std"]),    # [6, 1, 1]
    }


def global_normalize(feat: torch.Tensor, stats: dict) -> torch.Tensor:
    """
    Apply global normalisation. Must use IDENTICAL stats at train and inference.
    Preserves inter-file amplitude differences — unlike instance norm.
    """
    return (feat - stats["mean"]) / (stats["std"] + 1e-6)

## 🎭 Phase 7 — Training Augmentations
> All augmentations are **training only** — completely disabled in `infer.py`.
> EDA §6 addition: Pink noise alongside Gaussian to prevent overfitting to digital silence
> (dataset mean silence ratio = 86%).
> EDA §3 addition: time_shift restricted for Abnormal samples to avoid misaligning
> M1 onset events and M2 spin-up/down boundaries.

In [ ]:
def add_gaussian_noise(waveform: torch.Tensor,
                       snr_db_range: tuple = (15, 40)) -> torch.Tensor:
    """Additive white Gaussian noise at random SNR. Simulates mic quality variance."""
    snr_db  = torch.FloatTensor(1).uniform_(*snr_db_range).item()
    snr_lin = 10 ** (snr_db / 10)
    sig_power = waveform.pow(2).mean()
    noise_std = (sig_power / (snr_lin + 1e-9)).sqrt()
    return waveform + torch.randn_like(waveform) * noise_std


def add_pink_noise(waveform: torch.Tensor,
                   snr_db: float = 25.0) -> torch.Tensor:
    """
    Additive 1/f (pink) noise.
    EDA §6 requirement: prevents model from memorising pristine digital silence
    (dataset mean silence ratio = 86%, driven by M2).
    Pink noise has the same power spectral density as many real environments.
    """
    white      = torch.randn_like(waveform)
    # 1/f filter in frequency domain
    freqs      = torch.fft.rfftfreq(waveform.shape[-1]).clamp(min=1e-6)
    pink_filt  = (1.0 / freqs.sqrt())
    spectrum   = torch.fft.rfft(white)
    pink       = torch.fft.irfft(spectrum * pink_filt, n=waveform.shape[-1])
    # Normalise pink noise then scale to target SNR
    pink       = pink / (pink.abs().max() + 1e-9)
    sig_power  = waveform.pow(2).mean()
    snr_lin    = 10 ** (snr_db / 10)
    pink       = pink * (sig_power / (snr_lin + 1e-9)).sqrt()
    return waveform + pink


def random_gain(waveform: torch.Tensor,
                gain_range: tuple = (0.8, 1.2)) -> torch.Tensor:
    """Simulate recording volume variance between sessions."""
    gain = torch.FloatTensor(1).uniform_(*gain_range).item()
    return waveform * gain


def time_shift(waveform: torch.Tensor,
               max_shift_sec: float = 0.5) -> torch.Tensor:
    """
    Circular time shift.
    EDA §3 warning: use max_shift_sec=0.1 for Abnormal samples to avoid
    misaligning M1 onset events or M2 spin-up/down phases.
    Use max_shift_sec=0.5 (default) for Normal samples only.
    """
    shift = int(
        torch.FloatTensor(1).uniform_(-max_shift_sec, max_shift_sec).item()
        * TARGET_SR
    )
    return torch.roll(waveform, shifts=shift, dims=-1)


# SpecAugment transforms (built once)
_freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=24)
_time_mask = torchaudio.transforms.TimeMasking(time_mask_param=40)

def spec_augment(spec: torch.Tensor) -> torch.Tensor:
    """2 frequency masks + 2 time masks (pipeline.pdf §11.1)."""
    return _time_mask(_time_mask(_freq_mask(_freq_mask(spec))))


def mixup(spec_a: torch.Tensor, label_a: torch.Tensor,
          spec_b: torch.Tensor, label_b: torch.Tensor,
          alpha: float = 0.2):
    """Optional advanced augmentation — blends two samples (pipeline.pdf §11.5)."""
    lam        = np.random.beta(alpha, alpha)
    mixed_spec = lam * spec_a + (1 - lam) * spec_b
    mixed_lbl  = lam * label_a + (1 - lam) * label_b
    return mixed_spec, mixed_lbl

## ⚙️ Master Pipeline — `process_full()`
> This is the function the run loop calls. It was missing in the original notebook.

In [ ]:
def process_full(path: str,
                 is_training: bool = False,
                 is_abnormal: bool = False,
                 stats: dict = None) -> torch.Tensor:
    """
    Complete preprocessing pipeline: .wav → [6, 128, 344] tensor.

    Steps (in order):
      1. Load + stereo-fix (mono duplication)
      2. Resample to 16 kHz
      3. Waveform augmentations (training only)
      4. Pad / truncate to 176,000 samples (11s)
      5. Mel-spectrogram + dB → [2, 128, 344]
      6. Delta + ΔΔ features → [6, 128, 344]
      7. Global normalisation (if stats provided)
      8. SpecAugment (training only)

    Removed vs original:
      - DC offset removal     (EDA §8: DC = 0.000 on all files)
      - Peak normalisation    (EDA §4/8: destroys RMS fault signal & M3 transients)
      - Silence trimming      (EDA §6: destroys M2 temporal structure)
      - Per-file instance norm (EDA §4: erases inter-file energy differences)

    Args:
        path:        Path to .wav file.
        is_training: Enable waveform + spec augmentations.
        is_abnormal: Reduces time_shift max to 0.1s (EDA §3: protect boundary events).
        stats:       Global norm stats dict from load_global_stats(). If None, skips norm.

    Returns:
        Tensor of shape [6, 128, 344].
    """
    # Phase 1: Load
    waveform, sr = load_audio(path)

    # Phase 2: Resample to 16 kHz
    waveform = resample_waveform(waveform, sr)

    # Phase 3: Waveform augmentations (training only)
    if is_training:
        waveform = add_gaussian_noise(waveform)
        waveform = add_pink_noise(waveform)                    # EDA §6 addition
        waveform = random_gain(waveform)
        # EDA §3: restrict shift for abnormal to avoid misaligning boundary events
        shift_sec = 0.1 if is_abnormal else 0.5
        waveform  = time_shift(waveform, max_shift_sec=shift_sec)

    # Phase 4: Pad / truncate to exactly 11s
    waveform = fix_length(waveform)

    # Phase 5-6: Mel + dB + delta features → [6, 128, 344]
    feat = compute_features(waveform)

    # Phase 7: Global normalisation (EDA §4 override — NOT per-file instance norm)
    if stats is not None:
        feat = global_normalize(feat, stats)

    # Phase 8: SpecAugment (training only)
    if is_training:
        feat = spec_augment(feat)

    return feat

## ⚖️ Class Imbalance Handling
> **EDA §1 requirement.** The dataset has a 4.96:1 Normal/Abnormal imbalance.
> Both a `WeightedRandomSampler` and `WeightedCrossEntropyLoss` are needed.

In [ ]:
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler


def build_class_weights(df: pd.DataFrame,
                        num_classes: int = 6) -> torch.Tensor:
    """
    Compute per-class weights = 1 / class_count, normalised to sum to num_classes.
    Used for both WeightedRandomSampler and WeightedCrossEntropyLoss.

    EDA §1: Global imbalance 4.96:1 (Normal:Abnormal).
    Weights give Abnormal classes ~5x more importance.
    """
    counts  = df.groupby('label').size()                   # {0:16200, 1:3176, ...}
    weights = torch.zeros(num_classes)
    for label, count in counts.items():
        weights[label] = 1.0 / count
    weights = weights / weights.sum() * num_classes        # normalise
    return weights


def build_sampler(df: pd.DataFrame,
                  class_weights: torch.Tensor) -> WeightedRandomSampler:
    """
    WeightedRandomSampler assigns each sample a weight proportional to
    its class weight, so the DataLoader sees a roughly balanced distribution.
    """
    sample_weights = torch.tensor(
        [class_weights[label].item() for label in df['label']]
    )
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )


def build_criterion(class_weights: torch.Tensor,
                    device: str = 'cpu') -> nn.CrossEntropyLoss:
    """
    WeightedCrossEntropyLoss gives Abnormal classification errors ~5x more weight.
    EDA §1: critical for training on the 4.96:1 imbalanced dataset.
    """
    return nn.CrossEntropyLoss(weight=class_weights.to(device))


# Example usage (uncomment when building your DataLoader):
# class_weights = build_class_weights(df_train)
# sampler       = build_sampler(df_train, class_weights)
# criterion     = build_criterion(class_weights, device='cuda')
# loader = DataLoader(dataset, batch_size=32, sampler=sampler)

## ⚙️ Master Run Loop — Preprocessing & Saving
> Two-stage workflow:
> 1. **Stage A** — compute global stats from Normal train split (run once, saves `.npz`)
> 2. **Stage B** — preprocess all samples, save as `.npy` tensors

In [ ]:
def rebase_path(old_path: str, new_root: str) -> str:
    """Converts local Windows paths (D:\\...) to Kaggle/Linux paths."""
    import re
    parts = Path(old_path).parts
    for i, p in enumerate(parts):
        # Keep path starting from 'Machine 1', 'Machine 2', etc.
        if re.match(r'machine\s*\d', p, re.IGNORECASE):
            return os.path.join(new_root, *parts[i:])
    return os.path.join(new_root, os.path.basename(old_path))


# ── 1. Detect environment & set paths ─────────────────────────────────────────
IS_KAG = os.path.exists("/kaggle/input")
if IS_KAG:
    CSV_PATH   = "/kaggle/input/datat-set/dataset_audit.csv"
    DATA_ROOT  = "/kaggle/input/cmp27-machine-fault-recognition-full"
    SAMP_DIR   = "/kaggle/working/data/processed/"
    STATS_PATH = "/kaggle/working/norm_stats.npz"
    print(f"✅ Kaggle detected. Audio root: {DATA_ROOT}")
else:
    CSV_PATH   = "outputs/full_eda/dataset_audit.csv"
    SAMP_DIR   = "data/processed/"
    STATS_PATH = "norm_stats.npz"

# ── 2. Load CSV & rebase paths ────────────────────────────────────────────────
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"❌ CSV not found at {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
if IS_KAG:
    print("🔄 Rebasing paths from local → Kaggle...")
    df['path'] = df['path'].apply(lambda x: rebase_path(x, DATA_ROOT))

# Ensure numerical order within each group
df['file_id'] = df['filename'].str.extract(r'(\d+)').astype(int)
df = df.sort_values(['machine', 'state', 'file_id']).reset_index(drop=True)

os.makedirs(SAMP_DIR, exist_ok=True)
print(f"📂 Output directory: {SAMP_DIR}")
print(f"📊 Total files in audit CSV: {len(df):,}")

In [ ]:
# ── 3. Stage A: Compute global normalisation stats (run once) ─────────────────
# Only use the Normal training split to compute stats.
# If you have an explicit train/val/test split column, filter by split='train' too.

if not os.path.exists(STATS_PATH):
    print("📐 Computing global normalisation statistics from Normal training files...")
    df_normal_train = df[df['state'] == 'normal']
    # For speed during testing, set max_files=100. Use max_files=None for full run.
    global_stats = compute_global_stats(df_normal_train, save_path=STATS_PATH, max_files=None)
else:
    print(f"📐 Loading existing global stats from {STATS_PATH}")
    global_stats = load_global_stats(STATS_PATH)

print(f"   mean range: [{global_stats['mean'].min():.4f}, {global_stats['mean'].max():.4f}]")
print(f"   std  range: [{global_stats['std'].min():.4f},  {global_stats['std'].max():.4f}]")

In [ ]:
# ── 4. Stage B: Preprocess all samples & save as .npy ─────────────────────────
# Process ALL machines in one notebook (14 GB at [3, 128, 344] float16 < 20 GB limit)
df_proc = df.copy()
print(f"\n[ALL] Preprocessing {len(df_proc):,} total samples (all machines)...")
print(df_proc.groupby('label').size().rename('count').to_string())

errors = []
for i, row in tqdm(df_proc.iterrows(), total=len(df_proc), desc="Preprocessing"):
    dest = os.path.join(
        SAMP_DIR,
        f"{row['machine']}_{row['state']}_{Path(row['filename']).stem}.npy"
    )
    if os.path.exists(dest):
        continue
    try:
        is_abnormal = (row['state'] == 'abnormal')
        feat = process_full(
            path=row['path'],
            is_training=False,         # No augmentation baked in
            is_abnormal=is_abnormal,
            stats=global_stats,
        )
        np.save(dest, feat.half().numpy())   # float16 -> ~0.5 MB/file
    except Exception as e:
        errors.append((row['filename'], str(e)))
        if len(errors) <= 5:
            print(f"  ⚠️ Error on {row['filename']}: {e}")

print(f"\n✅ Preprocessing complete!")
print(f"   Saved to: {SAMP_DIR}")
print(f"   Tensor shape per file: [3, 128, 344]")
if errors:
    print(f"   ⚠️ {len(errors)} errors — check 'errors' list for details")

## ✅ Quick Sanity Check

In [ ]:
# Load one saved file and verify shape + value range
sample_files = list(Path(SAMP_DIR).glob("*.npy"))
if sample_files:
    arr = np.load(sample_files[0])
    print(f"Shape:  {arr.shape}   (expected [3, 128, 344])")
    print(f"dtype:  {arr.dtype}")
    print(f"min:    {arr.min():.4f}")
    print(f"max:    {arr.max():.4f}")
    print(f"mean:   {arr.mean():.4f}   (should be near 0 after global norm)")
    print(f"std:    {arr.std():.4f}    (should be near 1 after global norm)")

    # Visualise all 6 channels
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    titles = ["Mel", "Delta (Δ)", "Delta-Delta (ΔΔ)"]
    for idx, (ax, title) in enumerate(zip(axes.flat, titles)):
        ax.imshow(arr[idx], aspect='auto', origin='lower', cmap='magma')
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.suptitle(f"Sample: {sample_files[0].name}", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No .npy files found yet — run the preprocessing loop first.")

## 🚀 `infer.py` — Inference-Ready Pipeline
> Complete skeleton for the graded submission. Paste this into `infer.py`.
> Key compliance requirements:
> - Files processed in **strictly increasing numerical order** (pipeline.pdf §2.1)
> - `results.txt`: one integer per line, no headers
> - `time.txt`: one float per line, 3 decimal places, excludes I/O time

In [ ]:
INFER_PY = '''
import sys
import time
import torch
import numpy as np
from pathlib import Path

# ── Import pipeline functions (adjust import path as needed) ──────────────────
from preprocessing_fixed import (
    get_sorted_files,
    load_audio,
    resample_waveform,
    fix_length,
    compute_features,
    load_global_stats,
    global_normalize,
)


def preprocess_infer(path: str, stats: dict) -> torch.Tensor:
    """
    Inference-only pipeline: NO augmentations, NO silence trim, global norm only.
    Identical to process_full(..., is_training=False) but explicit for clarity.
    """
    waveform, sr = load_audio(path)
    waveform     = resample_waveform(waveform, sr)
    waveform     = fix_length(waveform)
    feat         = compute_features(waveform)
    feat         = global_normalize(feat, stats)
    return feat   # [6, 128, 344]


def main(data_dir: str):
    # Load model
    model = torch.load("model.pkl", map_location="cpu")
    model.eval()

    # Load global normalisation stats (same stats used at training time)
    stats = load_global_stats("norm_stats.npz")

    # CRITICAL: numerical sort (not alphabetical — pipeline.pdf §2.1)
    files = get_sorted_files(data_dir)

    results, times = [], []

    for wav_path in files:
        # Load audio (I/O — excluded from timer per spec)
        waveform, sr = load_audio(str(wav_path))

        # ---- START TIMER (after I/O) ----
        t_start = time.perf_counter()

        waveform = resample_waveform(waveform, sr)
        waveform = fix_length(waveform)
        feat     = compute_features(waveform)
        feat     = global_normalize(feat, stats)

        with torch.no_grad():
            logits = model(feat.unsqueeze(0))
            pred   = logits.argmax(dim=-1).item()

        # ---- STOP TIMER ----
        elapsed = round(time.perf_counter() - t_start, 3)

        results.append(pred)
        times.append(elapsed)

    # Write output files
    Path("results.txt").write_text("\n".join(map(str, results)) + "\n")
    Path("time.txt").write_text("\n".join(f"{t:.3f}" for t in times) + "\n")
    print(f"Done. Processed {len(results)} files.")


if __name__ == "__main__":
    main(sys.argv[1])
'''

with open("infer.py", "w") as f:
    f.write(INFER_PY.strip())
print("✅ infer.py written.")